## B.1 — Tensor, tích chập, weight sharing

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torchvision import models

torch.manual_seed(42)
count = lambda m: sum(p.numel() for p in m.parameters())
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())

x = torch.tensor([[4.,9.,2.,5.,8.,3.],[5.,6.,2.,4.,0.,3.],[2.,4.,5.,4.,5.,2.],
                  [5.,6.,5.,4.,7.,8.],[5.,7.,7.,9.,2.,1.],[5.,8.,5.,3.,8.,4.]]).reshape(1,1,6,6)
k = torch.tensor([[1.,0.,-1.],[1.,0.,-1.],[1.,0.,-1.]]).reshape(1,1,3,3)
y = F.conv2d(x, k)
assert y.shape == (1,1,4,4) and y[0,0,0,0].item() == 2
print('Input:', tuple(x.shape), '| Output:', tuple(y.shape), '| Giá trị đầu:', y[0,0,0,0].item())
print('MLP 64×64×3 -> 64×64×16:', 64*64*3*64*64*16)
print('Conv 3 -> 16, kernel 3×3:', count(nn.Conv2d(3,16,3,padding=1)))

PyTorch: 2.11.0+cu128 | GPU: True
Input: (1, 1, 6, 6) | Output: (1, 1, 4, 4) | Giá trị đầu: 2.0
MLP 64×64×3 -> 64×64×16: 805306368
Conv 3 -> 16, kernel 3×3: 448


**Nhận xét:** Conv dùng chung trọng số giữa các vị trí; số tham số không tăng theo chiều dài/rộng của ảnh.

## B.2 — Output size, pooling, receptive field

In [2]:
def out_size(i,k,s=1,p=0,d=1):
    return (i+2*p-d*(k-1)-1)//s+1

print('Conv(28, K=3, S=2, P=1):', out_size(28,3,2,1))
a = torch.rand(1,64,16,16)
print('MaxPool:', tuple(nn.MaxPool2d(2)(a).shape), '| AvgPool:', tuple(nn.AvgPool2d(2)(a).shape))
print('Receptive field của 1/2/3 Conv 3×3 stride 1:', [2*i+1 for i in (1,2,3)])

cnn = nn.Sequential(nn.Conv2d(3,10,3),nn.ReLU(),nn.Conv2d(10,20,5,2),nn.ReLU(),
                    nn.Conv2d(20,40,5,2),nn.ReLU())
a = torch.rand(1,3,39,39)
for layer in cnn:
    a = layer(a)
    if isinstance(layer, nn.Conv2d): print(layer.__class__.__name__, tuple(a.shape))
print('Flatten:', a.flatten(1).shape)
assert a.flatten(1).shape[1] == 1960

Conv(28, K=3, S=2, P=1): 14
MaxPool: (1, 64, 8, 8) | AvgPool: (1, 64, 8, 8)
Receptive field của 1/2/3 Conv 3×3 stride 1: [3, 5, 7]
Conv2d (1, 10, 37, 37)
Conv2d (1, 20, 17, 17)
Conv2d (1, 40, 7, 7)
Flatten: torch.Size([1, 1960])


## B.3 — LeNet-5 (64×64×3, 21 lớp)

In [3]:
class LeNet5(nn.Module):
    def __init__(self, pool='avg', img=64):
        super().__init__()
        Pool = nn.AvgPool2d if pool=='avg' else nn.MaxPool2d
        self.c1, self.s2 = nn.Conv2d(3,6,5), Pool(2)
        self.c3, self.s4 = nn.Conv2d(6,16,5), Pool(2)
        with torch.no_grad():
            z = self.s4(self.c3(self.s2(self.c1(torch.zeros(1,3,img,img)))))
        self.f5 = nn.Linear(z.flatten(1).shape[1],120)
        self.f6 = nn.Linear(120,84)
        self.out = nn.Linear(84,21)
    def forward(self,x):
        x = self.s2(F.relu(self.c1(x)))
        x = self.s4(F.relu(self.c3(x)))
        return self.out(F.relu(self.f6(F.relu(self.f5(x.flatten(1))))))

net = LeNet5()
shapes, x = [], torch.zeros(1,3,64,64)
for name in ['c1','s2','c3','s4']:
    layer = getattr(net,name)
    x = layer(x)
    shapes.append([name,tuple(x.shape[1:]),count(layer)])
x = x.flatten(1)
for name in ['f5','f6','out']:
    layer = getattr(net,name)
    x = layer(x)
    shapes.append([name,tuple(x.shape[1:]),count(layer)])
display(pd.DataFrame(shapes, columns=['Layer','Output shape','Tham số']))
print('Tổng:',count(net),'| 32×32:',count(LeNet5(img=32)))
assert count(net)==339421 and count(LeNet5(img=32))==62941

,Layer,Output shape,Tham số
0,c1,"(6, 60, 60)",456
1,s2,"(6, 30, 30)",0
2,c3,"(16, 26, 26)",2416
3,s4,"(16, 13, 13)",0
4,f5,"(120,)",324600
5,f6,"(84,)",10164
6,out,"(21,)",1785


Tổng: 339421 | 32×32: 62941


## B.4 — Inception block, nối channel và GAP

In [4]:
class Inception(nn.Module):
    def __init__(self,c,f1,f3r,f3,f5r,f5,fp):
        super().__init__()
        self.b1=nn.Sequential(nn.Conv2d(c,f1,1),nn.ReLU())
        self.b2=nn.Sequential(nn.Conv2d(c,f3r,1),nn.ReLU(),nn.Conv2d(f3r,f3,3,padding=1),nn.ReLU())
        self.b3=nn.Sequential(nn.Conv2d(c,f5r,1),nn.ReLU(),nn.Conv2d(f5r,f5,5,padding=2),nn.ReLU())
        self.b4=nn.Sequential(nn.MaxPool2d(3,1,1),nn.Conv2d(c,fp,1),nn.ReLU())
    def forward(self,x):
        return torch.cat([self.b1(x),self.b2(x),self.b3(x),self.b4(x)],dim=1)

block = Inception(1,16,16,24,4,8,8)
out = block(torch.rand(1,1,28,28))
print('Inception:',tuple(out.shape),'| Tham số:',count(block))
print('GAP:',tuple(nn.AdaptiveAvgPool2d(1)(out).shape))
assert out.shape==(1,56,28,28) and count(block)==4376

Inception: (1, 56, 28, 28) | Tham số: 4376
GAP: (1, 56, 1, 1)


## B.5 — Residual block và ResNet-18

In [5]:
class BasicBlock(nn.Module):
    def __init__(self,ci,co,stride=1):
        super().__init__()
        self.conv1=nn.Conv2d(ci,co,3,stride,1,bias=False)
        self.bn1=nn.BatchNorm2d(co)
        self.conv2=nn.Conv2d(co,co,3,1,1,bias=False)
        self.bn2=nn.BatchNorm2d(co)
        self.shortcut=nn.Identity() if ci==co and stride==1 else nn.Sequential(
            nn.Conv2d(ci,co,1,stride,bias=False),nn.BatchNorm2d(co))
    def forward(self,x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))+self.shortcut(x))

print('Identity:',tuple(BasicBlock(64,64)(torch.rand(1,64,16,16)).shape))
print('Projection:',tuple(BasicBlock(64,128,2)(torch.rand(1,64,16,16)).shape))
r18=models.resnet18(weights=None,num_classes=21)
print('ResNet-18:',count(r18),'tham số')
assert count(r18)==11187285

Identity: (1, 64, 16, 16)
Projection: (1, 128, 8, 8)
ResNet-18: 11187285 tham số


## B.6 — ResNet-50: classifier và đóng băng backbone

In [6]:
r50=models.resnet50(weights=None,num_classes=21)
for p in r50.parameters(): p.requires_grad=False
for p in r50.fc.parameters(): p.requires_grad=True
print('Tổng tham số:', count(r50))
print('Feature extraction, tham số được học:',sum(p.numel() for p in r50.parameters() if p.requires_grad))
assert sum(p.numel() for p in r50.parameters() if p.requires_grad)==43029

Tổng tham số: 23551061
Feature extraction, tham số được học: 43029


B.6 ở đây **chỉ kiểm tra cấu trúc** bằng `torchvision` (không tải pretrained). Fine-tuning pretrained HuggingFace, nếu chọn làm bài nâng cao, ở Notebook 02.

## B.7 — Minh họa 4 độ đo (không phải kết quả VinaFood21)

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
true=[0,0,1,1,2,2]
pred=[0,1,1,1,2,0]
p,r,f,_=precision_recall_fscore_support(true,pred,average='macro',zero_division=0)
print({'accuracy':accuracy_score(true,pred),'precision_macro':p,'recall_macro':r,'f1_macro':f})
print('Kết quả thật của VinaFood21 được tạo ở Notebook 02.')

{'accuracy': 0.6666666666666666, 'precision_macro': 0.7222222222222222, 'recall_macro': 0.6666666666666666, 'f1_macro': 0.6555555555555556}
Kết quả thật của VinaFood21 được tạo ở Notebook 02.
